# Train the misconception-tagged AP Bio item generator (QLoRA)

Fine-tunes **Qwen/Qwen3-1.7B** on the by-construction generation dataset to instill the Behavior Spec
(`docs/behavior_spec.md`). Runtime: **Runtime > Change runtime type > T4/A100 GPU**.

Pipeline: install → get code+data → train (`scripts/train_gen_sft.py`) → sanity-generate →
(optional) push adapter to HF Hub → base-vs-tuned eval.

**Before running:** commit + push the repo (including `data/gen_sft_train.jsonl`,
`data/gen_sft_val.jsonl`, `data/eval_scenarios.jsonl`, `data/apbio_misconceptions.json`) so the clone
cell can fetch them — or use the upload fallback cell.

In [ ]:
!nvidia-smi -L  # confirm a GPU is attached

In [ ]:
# Unsloth QLoRA stack (~2x faster, ~70% less VRAM; fits a free T4).
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl datasets

In [ ]:
# Get code + data. Set REPO to your fork's URL; the repo must already contain
# the data/*.jsonl files (commit them first).
REPO = "https://github.com/gabriel-xiong/algebra_error_classifier.git"
import os
if not os.path.isdir("algebra_error_classifier"):
    !git clone $REPO
%cd algebra_error_classifier
!ls data/gen_sft_train.jsonl data/gen_sft_val.jsonl data/eval_scenarios.jsonl

### Upload fallback (only if the clone lacks the data files)
Run the next cell to upload `gen_sft_train.jsonl`, `gen_sft_val.jsonl`, `eval_scenarios.jsonl`,
and `apbio_misconceptions.json` from your machine into `data/`.

In [ ]:
# from google.colab import files
# import shutil, os
# os.makedirs('data', exist_ok=True)
# up = files.upload()
# for name in up:
#     shutil.move(name, f'data/{name}')

In [ ]:
# Train. Completion-only masking is applied inside the script (loss on the JSON only).
!python scripts/train_gen_sft.py \
    --model Qwen/Qwen3-1.7B \
    --train data/gen_sft_train.jsonl \
    --val   data/gen_sft_val.jsonl \
    --out   outputs/gen_lora \
    --epochs 3 --batch-size 4 --grad-accum 4 --max-seq-length 1536

In [ ]:
# Sanity check: load base+adapter and generate one item from a held-out scenario.
import json, sys
sys.path.insert(0, 'scripts')
from unsloth import FastLanguageModel
import gen_spec

model, tok = FastLanguageModel.from_pretrained('outputs/gen_lora', max_seq_length=1536, load_in_4bit=True)
FastLanguageModel.for_inference(model)

sc = json.loads(open('data/eval_scenarios.jsonl').readline())
system, user = gen_spec.build_generation_prompt(sc['topic'], sc['misconception_ids'])
msgs = [{'role':'system','content':system},{'role':'user','content':user}]
text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
ids = tok(text, return_tensors='pt').to(model.device)
out = model.generate(**ids, max_new_tokens=512, do_sample=False, pad_token_id=tok.eos_token_id)
print(tok.decode(out[0][ids['input_ids'].shape[1]:], skip_special_tokens=True))

In [ ]:
# (Optional) Push the LoRA adapter to the HF Hub for the submission.
# from huggingface_hub import login; login()  # paste an HF write token
# model.push_to_hub('YOUR_USER/apbio-item-generator-qwen3-1.7b-lora')
# tok.push_to_hub('YOUR_USER/apbio-item-generator-qwen3-1.7b-lora')

### Base-vs-tuned eval (the results table)
Genetics is scored objectively offline (no key). The **conceptual** dimensions need the calibrated
judge — set `OPENAI_API_KEY` in the cell below and pass `--judge-model gpt-4o` (gpt-4o-mini FAILED
calibration). Merge the LoRA into a full model first so `eval_generation.py`'s HF loader can read it.

In [ ]:
# Merge adapter -> standalone model dir that eval_generation.py can load with hf:
model.save_pretrained_merged('outputs/gen_merged', tok, save_method='merged_16bit')

import os
os.environ['OPENAI_API_KEY'] = ''  # <-- paste your key (needed only for conceptual scoring)

!python scripts/eval_generation.py \
    --base  hf:Qwen/Qwen3-1.7B \
    --tuned hf:outputs/gen_merged \
    --judge-model gpt-4o \
    --out data/eval_results.jsonl